# BENZI — LoRA fine-tune on Colab (~2–3 hours)

**Before you start:** *Runtime → Change runtime type → **T4 GPU***

**Every run:** *Runtime → **Disconnect and delete runtime***, then open this notebook from GitHub and *Run all*.

| Step | What | Time |
|------|------|------|
| Setup | Clone + install | ~3 min |
| Dataset | Empathy JSONL | ~5 min |
| Train | Qwen2.5-3B LoRA | ~45–90 min |
| Merge + zip | Download for Mac | ~15 min |

After download see `benzi-server/docs/COLAB_TRAIN.md`.


In [52]:
# GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable T4 GPU: Runtime → Change runtime type → GPU"
print("CUDA OK")


Sun May 31 15:43:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [55]:
import os, subprocess
from pathlib import Path

os.chdir("/content")
!rm -rf /content/final-year-benzi /content/final-year-benzi-tmp

r = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/sameedsaeed123/final-year-benzi.git",
     "/content/final-year-benzi"],
    capture_output=True, text=True,
)
print(r.stderr or r.stdout or "clone OK")
assert r.returncode == 0, "Clone failed — try Runtime → Disconnect and delete runtime"

%cd /content/final-year-benzi/fyp-ml-demos
!sed -i '/use_mps_device/d' finetune/train_qlora.py 2>/dev/null || true
print("Ready:", Path.cwd())

Cloning into '/content/final-year-benzi'...

/content/final-year-benzi/fyp-ml-demos
Ready: /content/final-year-benzi/fyp-ml-demos


In [ ]:
# Install deps (Colab: keep numpy 2.x — do not use requirements-finetune.txt)
import subprocess, sys
from pathlib import Path

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-r", "requirements.txt")
colab = Path("requirements-finetune-colab.txt")
if colab.is_file():
    pip("-r", str(colab))
else:
    pip("datasets>=2.19.0", "peft>=0.11.0", "accelerate>=0.30.0",
        "sentencepiece>=0.2.0", "protobuf>=4.25.0", "tqdm>=4.66.0")
pip("bitsandbytes>=0.43.0", "accelerate")

import numpy as np, bitsandbytes as bnb, torch
print("numpy", np.__version__, "| bitsandbytes", bnb.__version__)


In [56]:
# Step 1 — dataset (~5 min)
!python finetune/prepare_dataset.py --max-total 1200


[empathetic] loading up to 600…
[empathetic] got 600 rows
[counsel-chat] loading up to 400…
Repo card metadata block was not found. Setting CardData to empty.
[counsel-chat] got 400 rows
Wrote /content/final-year-benzi/fyp-ml-demos/finetune/data/benzi_train.jsonl (950 rows)
Wrote /content/final-year-benzi/fyp-ml-demos/finetune/data/benzi_val.jsonl (50 rows)
Stats: /content/final-year-benzi/fyp-ml-demos/finetune/data/benzi_dataset_stats.json


In [57]:
# Step 2 — train (~45–90 min). Wait until you see: Saved LoRA adapter
!python finetune/train_qlora.py --model Qwen/Qwen2.5-3B-Instruct --max-steps 60 --max-length 384 --batch-size 1 --grad-accum 4


Training on 950 examples, max_steps=60, model=Qwen/Qwen2.5-3B-Instruct
Device: cuda, 4bit: True
Loading weights:   1% 3/434 [00:01<05:58,  1.20it/s, Materializing param=model.layers.0.mlp.down_proj.weight]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 434/434 [00:15<00:00, 27.43it/s, Materializing param=model.norm.weight] 
trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
Map: 100% 950/950 [00:00<00:00, 1344.78 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Starting training…
  0% 0/60 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 

In [61]:
%cd /content/final-year-benzi/fyp-ml-demos

!pip uninstall -y torchao
!python finetune/merge_lora.py --model Qwen/Qwen2.5-3B-Instruct

/content/final-year-benzi/fyp-ml-demos
Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0
Loading base Qwen/Qwen2.5-3B-Instruct…
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:25<00:00, 16.86it/s, Materializing param=model.norm.weight]
Merging LoRA weights…
Writing model shards:   0% 0/1 [00:00<?, ?it/s]^C


**Faster option (~20 min):** comment out Step 2 above and run:
```python
!python finetune/train_qlora.py --benzi-lite
```


In [60]:
# Step 3 — merge (~10 min) — only after Step 2 succeeds
from pathlib import Path
adapter_cfg = Path("finetune/adapters/benzi-lora/adapter_config.json")
if not adapter_cfg.is_file():
    raise RuntimeError(
        "Training did not finish — no adapter_config.json.\n"
        "Re-run Step 2 and wait for 'Saved LoRA adapter' before merge."
    )
!python finetune/merge_lora.py --model Qwen/Qwen2.5-3B-Instruct


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading base Qwen/Qwen2.5-3B-Instruct…
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:25<00:00, 16.84it/s, Materializing param=model.norm.weight]
Merging LoRA weights…
Writing model shards:   0% 0/1 [00:00<?, ?it/s]^C


In [ ]:
# Step 4 — download zip
import shutil
from pathlib import Path
from google.colab import files

merged = Path("finetune/merged/benzi-empathetic-hf")
if not merged.is_dir():
    raise RuntimeError("Merged model missing — complete Step 3 first.")

shutil.make_archive("/content/benzi-empathetic-trained", "zip", merged)
files.download("/content/benzi-empathetic-trained.zip")
print("Download started: benzi-empathetic-trained.zip")


## On your Mac

```bash
mkdir -p ~/benzi-models && cd ~/benzi-models
unzip ~/Downloads/benzi-empathetic-trained.zip -d benzi-empathetic-hf
cd benzi-empathetic-hf
cat > Modelfile << 'EOF'
FROM .
PARAMETER temperature 0.65
PARAMETER num_ctx 4096
SYSTEM You are BENZI AI — supportive wellness between therapy sessions. Not a therapist. Defer clinical questions to their therapist.
EOF
ollama create benzi-empathetic-trained -f Modelfile
```

In `benzi-server/.env`: `OLLAMA_MODEL=benzi-empathetic-trained` then restart the API.
